To describe convective-diffusive problems, the MRT lattice Boltzmann formulation is described by:

$$
f_i( x_{\alpha} + e_{i,\alpha} \delta t, t+\delta t) = f_i(x_{\alpha}, t)  - (\textbf{M}^{-1}\textbf{S}\textbf{M})_{ik}\left( f_{k} - f_{k}^{eq}  \right), 
$$(MRT-LB-conv-df-Eq)

where $\textbf{M}$ is matrix of moments, $\textbf{S}$ is the diagonal matrix of relaxation times, and $\textbf{M}^{-1}$ is the inverse matrix of $\textbf{M}$. Considering the problem description in a lattice D1Q3, we have:

$$
\textbf{M} =\begin{pmatrix} 1 \\ e_{i,x} \\ e_{i,x}^{2}-c_{s}^{2} \end{pmatrix} =\begin{pmatrix} 1 & 1 & 1 \\ 0 & 1   & -1 \\ -c_{s}^{2}  & 1-c_{s}^{2} & 1-c_{s}^{2} \end{pmatrix},  \qquad \qquad  \textbf{S} = \begin{pmatrix} s_0 & 0   & 0 \\ 0   & s_1 & 0 \\ 0   & 0   & s_2 \end{pmatrix},  \qquad \qquad \textbf{M}^{-1}=\begin{pmatrix}1-c_{s}^{2} & 0 & -1 \\ \dfrac{c_{s}^{2}}{2} & \dfrac{1}{2} & \dfrac{1}{2} \\ \dfrac{c_{s}^{2}}{2} & -\dfrac{1}{2} & \dfrac{1}{2} \end{pmatrix}.
$$

The equilibrium distribution function is defined by

$$
f^{eq}_{i} = w_{i}\left( u \right).
$$


In [1]:
# import warnings
# warnings.filterwarnings("ignore")
# from pylab import *
# from __future__ import division
# from sympy import *
# import numpy as np
# from sympy import S, collect, expand, factor, Wild
# from sympy import fraction, Rational, Symbol
# from sympy import symbols, sqrt, Rational
# import sympy as sp
# from IPython.display import display, Math, Latex
# #-------------------------------------------------Símbolos----------------------------------------------
# omega, u, B, w, C, Lamb = symbols('omega, u, B_{\\alpha}, w, C, \\Lambda_{\\alpha\\beta}')
# wi, cx, cy, cs = symbols('w_{i} c_{x} c_{y} c_{s}')
# fi, f0, f1, f2  = symbols('f_{i} f_{0} f_{1} f_{2}')
# #-------------------------------------------------Funções----------------------------------------------
# feq = Function('feq')(wi, cx, cy)
# fneq = Function('fneq')(wi, cx, cy)
# f = Function('f')(fi)
# #----------------------------------------------Lattice-D2Q9---Variáveis----------------------------------------------
# fi=np.array([f0,f1,f2])
# w0=Rational(4,6);w1=Rational(1,6)
# wi=np.array([w0,w1,w1])
# cx=np.array([0,1,-1])
# as2=Rational(3)
# cs2=1/as2
# #-------------------------------------------------Calc.Func------------------------------------------------
# f= fi
# feq=wi*(u)

In [2]:
import warnings
warnings.filterwarnings("ignore")
from pylab import *
from sympy import *
import numpy as np

w0, w1, w2 = symbols(['w_0','w_1','w_2'])
us = symbols('u_{x}^{t}')
Tc0, Tc1, Tc2= symbols([r'T_{\Delta_{x}}^{c_{0}}','T_{\Delta_{x}}^{c_{1}}','T_{\Delta_{x}}^{c_{2}}'])
cx=np.array([0,1,-1])
wis=np.array([w0,w1,w2])
omega = symbols(r'\omega')
cs = symbols('c_s')
f0, f1, f2, f3, f4 = symbols(['f_{0\,x}^t','f_{1\,x}^t','f_{2\,x}^t','f_{3\,x}^t','f_{4\,x}^t'])
I=eye(3)
Tc = Matrix([I[0,:]*Tc0,I[1,:]*Tc1,I[2,:]*Tc2])
Tcl = Matrix([Tc0,Tc1,Tc2])
feq = us*wis
feqM = Matrix([feq[0],feq[1],feq[2]])
fM = Matrix([f0,f1,f2])
# m0 = np.array([1,1,1])
# m1 = cx
# m2 = cx*cx - cs*cs
# MM = Matrix([m0,m1,m2])
# MMinv = MM.inv()

In [3]:
import sympy as sp
p = {}
past_steps = [1, 2, 3, 4]
space_offsets = [1, 2, 3]
# time_map: key_prefix -> (latex_expr)
#   "+1" -> u^{t+1},  "0" -> u^{t},  "-1".."-4" -> u^{t-k}
time_map = {"+1": "t+1","0":  "t",**{f"-{k}": f"t-{k}" for k in past_steps},}

# Non-shifted symbols
for tk, te in time_map.items():
    p[f"ut{tk}"] = sp.Symbol(fr'u^{{{te}}}_{{x}}')

# Shifted symbols (skip the future step "+1")
for tk, te in time_map.items():
    if tk == "+1":
        continue
    for ssign in ("+", "-"):
        for d in space_offsets:
            p[f"ut{tk}d{ssign}{d}"] = sp.Symbol(fr'u^{{{te}}}_{{x{ssign}{d}}}')

In [4]:
p

{'ut+1': u^{t+1}_{x},
 'ut0': u^{t}_{x},
 'ut-1': u^{t-1}_{x},
 'ut-2': u^{t-2}_{x},
 'ut-3': u^{t-3}_{x},
 'ut-4': u^{t-4}_{x},
 'ut0d+1': u^{t}_{x+1},
 'ut0d+2': u^{t}_{x+2},
 'ut0d+3': u^{t}_{x+3},
 'ut0d-1': u^{t}_{x-1},
 'ut0d-2': u^{t}_{x-2},
 'ut0d-3': u^{t}_{x-3},
 'ut-1d+1': u^{t-1}_{x+1},
 'ut-1d+2': u^{t-1}_{x+2},
 'ut-1d+3': u^{t-1}_{x+3},
 'ut-1d-1': u^{t-1}_{x-1},
 'ut-1d-2': u^{t-1}_{x-2},
 'ut-1d-3': u^{t-1}_{x-3},
 'ut-2d+1': u^{t-2}_{x+1},
 'ut-2d+2': u^{t-2}_{x+2},
 'ut-2d+3': u^{t-2}_{x+3},
 'ut-2d-1': u^{t-2}_{x-1},
 'ut-2d-2': u^{t-2}_{x-2},
 'ut-2d-3': u^{t-2}_{x-3},
 'ut-3d+1': u^{t-3}_{x+1},
 'ut-3d+2': u^{t-3}_{x+2},
 'ut-3d+3': u^{t-3}_{x+3},
 'ut-3d-1': u^{t-3}_{x-1},
 'ut-3d-2': u^{t-3}_{x-2},
 'ut-3d-3': u^{t-3}_{x-3},
 'ut-4d+1': u^{t-4}_{x+1},
 'ut-4d+2': u^{t-4}_{x+2},
 'ut-4d+3': u^{t-4}_{x+3},
 'ut-4d-1': u^{t-4}_{x-1},
 'ut-4d-2': u^{t-4}_{x-2},
 'ut-4d-3': u^{t-4}_{x-3}}

In [5]:
feqM

Matrix([
[u_{x}^{t}*w_0],
[u_{x}^{t}*w_1],
[u_{x}^{t}*w_2]])

In [6]:
omega*feqM.T*Tc 

Matrix([[T_{\Delta_{x}}^{c_{0}}*\omega*u_{x}^{t}*w_0, T_{\Delta_{x}}^{c_{1}}*\omega*u_{x}^{t}*w_1, T_{\Delta_{x}}^{c_{2}}*\omega*u_{x}^{t}*w_2]])

In [7]:
Tcl*us

Matrix([
[T_{\Delta_{x}}^{c_{0}}*u_{x}^{t}],
[T_{\Delta_{x}}^{c_{1}}*u_{x}^{t}],
[T_{\Delta_{x}}^{c_{2}}*u_{x}^{t}]])

In [8]:
O1=Add(*list( omega*Tc*feqM + (1-omega)*Tcl*us ) )
O1_2= sp.collect(O1,[Tc0*us,Tc1*us,Tc2*us])
O1_2

T_{\Delta_{x}}^{c_{0}}*u_{x}^{t}*(\omega*w_0 - \omega + 1) + T_{\Delta_{x}}^{c_{1}}*u_{x}^{t}*(\omega*w_1 - \omega + 1) + T_{\Delta_{x}}^{c_{2}}*u_{x}^{t}*(\omega*w_2 - \omega + 1)

In [9]:
Add(*list( omega*Tc*feqM ))

T_{\Delta_{x}}^{c_{0}}*\omega*u_{x}^{t}*w_0 + T_{\Delta_{x}}^{c_{1}}*\omega*u_{x}^{t}*w_1 + T_{\Delta_{x}}^{c_{2}}*\omega*u_{x}^{t}*w_2

In [10]:
OrderT = Matrix([0,0,0])
for i0 in range (3):
    print(i0)
    display(Tcl[i0])
    OrderT[0]=OrderT[0] + omega*feqM[i0]*Tcl[i0] + (1-omega)*Tcl[i0]*us 
    for i1 in range (3):
        print(i0,i1)
        if (i1!=i0):
            display(Tcl[i0]*Tcl[i1])
            OrderT[1]=OrderT[1] + omega*feqM[i1]*Tcl[i0]*Tcl[i1] + (1-omega)*Tcl[i0]*Tcl[i1]*us 
        for i2 in range (3):
            print(i0,i1,i2)
            if (i2!=i1 and i2!=i0 and i1!=i0):
                display(Tcl[i0]*Tcl[i1]*Tcl[i2])
                OrderT[2]=OrderT[2] + omega*feqM[i2]*Tcl[i0]*Tcl[i1]*Tcl[i2] + (1-omega)*Tcl[i0]*Tcl[i1]*Tcl[i2]*fM[i2] 

0


T_{\Delta_{x}}^{c_{0}}

0 0
0 0 0
0 0 1
0 0 2
0 1


T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}

0 1 0
0 1 1
0 1 2


...

2 0 2
2 1


T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}

2 1 0


T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}

2 1 1
2 1 2
2 2
2 2 0
2 2 1
2 2 2


In [11]:
OrderT[0]

T_{\Delta_{x}}^{c_{0}}*\omega*u_{x}^{t}*w_0 + T_{\Delta_{x}}^{c_{0}}*u_{x}^{t}*(1 - \omega) + T_{\Delta_{x}}^{c_{1}}*\omega*u_{x}^{t}*w_1 + T_{\Delta_{x}}^{c_{1}}*u_{x}^{t}*(1 - \omega) + T_{\Delta_{x}}^{c_{2}}*\omega*u_{x}^{t}*w_2 + T_{\Delta_{x}}^{c_{2}}*u_{x}^{t}*(1 - \omega)

In [12]:
sp.collect(OrderT[0]
           .subs({Tc1*Tc2:1,Tc0:1})
           .subs(Tc1*us,p['ut0d-1'])
           .subs(Tc2*us,p['ut0d+1'])
           # .subs({w1:Rational(1,6),w0:Rational(2,3)})
           ,{us,p['ut0d-1'],p['ut0d+1']}
          )

u^{t}_{x+1}*(\omega*w_2 - \omega + 1) + u^{t}_{x-1}*(\omega*w_1 - \omega + 1) + u_{x}^{t}*(\omega*w_0 - \omega + 1)

In [13]:
OrderT[1]

T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*\omega*u_{x}^{t}*w_0 + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*\omega*u_{x}^{t}*w_1 + 2*T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*u_{x}^{t}*(1 - \omega) + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{2}}*\omega*u_{x}^{t}*w_0 + T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{2}}*\omega*u_{x}^{t}*w_2 + 2*T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{2}}*u_{x}^{t}*(1 - \omega) + T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}*\omega*u_{x}^{t}*w_1 + T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}*\omega*u_{x}^{t}*w_2 + 2*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}*u_{x}^{t}*(1 - \omega)

In [14]:
sp.simplify(sp.collect(OrderT[2]
           .subs({Tc1*Tc2:1,Tc0:1})
           .subs({w1:Rational(1,6),w2:Rational(1,6),w0:Rational(2,3)})
           ,(1-omega)
          ).subs(2*f0+2*f1+2*f2,2*us))/2

u_{x}^{t}

In [15]:
sp.collect(OrderT[1]
           .subs({Tc1*Tc2:1,Tc0:1})
           .subs(Tc1*us,p['ut-1d-1'])
           .subs(Tc2*us,p['ut-1d+1'])
           # .subs({w1:Rational(1,6),w0:Rational(2,3)})
           ,{p['ut-1d-1'],p['ut-1d+1']}
          )

\omega*u_{x}^{t}*w_1 + \omega*u_{x}^{t}*w_2 + u^{t-1}_{x+1}*(\omega*w_0 + \omega*w_2 - 2*\omega + 2) + u^{t-1}_{x-1}*(\omega*w_0 + \omega*w_1 - 2*\omega + 2) + 2*u_{x}^{t}*(1 - \omega)

In [16]:
OrderT = Matrix([0, 0, 0])

# First order (single index)
for i0 in range(3):
    print(i0)
    display(Tcl[i0])
    OrderT[0] = OrderT[0] + omega * feqM[i0] * Tcl[i0] + (1 - omega) * Tcl[i0] * us

# Second order (unordered pairs i0 < i1)
for i0 in range(3):
    for i1 in range(i0 + 1, 3):
        print(i0, i1)
        display(Tcl[i0] * Tcl[i1])
        OrderT[1] = OrderT[1] + omega * feqM[i1] * Tcl[i0] * Tcl[i1] \
                    + (1 - omega) * Tcl[i0] * Tcl[i1] * us

# Third order (unordered triples i0 < i1 < i2)
for i0 in range(3):
    for i1 in range(i0 + 1, 3):
        for i2 in range(i1 + 1, 3):
            print(i0, i1, i2)
            display(Tcl[i0] * Tcl[i1] * Tcl[i2])
            OrderT[2] = OrderT[2] + omega * feqM[i2] * Tcl[i0] * Tcl[i1] * Tcl[i2] \
                        + (1 - omega) * Tcl[i0] * Tcl[i1] * Tcl[i2] * fM[i2]

0


T_{\Delta_{x}}^{c_{0}}

1


T_{\Delta_{x}}^{c_{1}}

2


...

T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{2}}

1 2


T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}

0 1 2


T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}

In [17]:
sp.collect(OrderT[0]
           .subs({Tc1*Tc2:1,Tc0:1})
           .subs(Tc1*us,p['ut0d-1'])
           .subs(Tc2*us,p['ut0d+1'])
           # .subs({w1:Rational(1,6),w0:Rational(2,3)})
           ,{us,p['ut0d-1'],p['ut0d+1']}
          )

u^{t}_{x+1}*(\omega*w_2 - \omega + 1) + u^{t}_{x-1}*(\omega*w_1 - \omega + 1) + u_{x}^{t}*(\omega*w_0 - \omega + 1)

In [16]:
sp.collect(OrderT[1]
           .subs({Tc1*Tc2:1,Tc0:1})
           .subs(Tc1*us,p['ut-1d-1'])
           .subs(Tc2*us,p['ut-1d+1'])
           # .subs({w1:Rational(1,6),w0:Rational(2,3)})
           ,{p['ut-1d-1'],p['ut-1d+1']}
          )

\omega*u_{x}^{t}*w_2 + u^{t-1}_{x+1}*(\omega*w_2 - \omega + 1) + u^{t-1}_{x-1}*(\omega*w_1 - \omega + 1) + u_{x}^{t}*(1 - \omega)

In [19]:
sp.simplify(sp.collect(OrderT[2]
           # .subs({Tc1*Tc2:1,Tc0:1})
           # .subs({w1:Rational(1,6),w2:Rational(1,6),w0:Rational(2,3)})
           ,(1-omega)
          ).subs(2*f0+2*f1+2*f2,2*us))/2

T_{\Delta_{x}}^{c_{0}}*T_{\Delta_{x}}^{c_{1}}*T_{\Delta_{x}}^{c_{2}}*(\omega*u_{x}^{t}*w_2 - f_{2,x}^t*(\omega - 1))/2